In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

data = pd.read_csv(r"C:\Users\jcwvi\Desktop\INaturalist Project\INaturalistDataset.csv")
introduced_sp = pd.read_csv("Introduced_Sp.csv")

<h3>Basic Data Cleaning: Ensuring binomial nomenclature formatting and changing absent fields to pd.NA </h3>
With INaturalist observations, all entries are user-made. This makes for a wide variety of languages, grammar and punctuation mistakes, and errors in values. Fortunately, fields like taxon_name and preferred_common_name are generated by INaturalist, and are standardized.<br>
<br>
Additionally, I took the liberty of altering the establishment means of taxa whose name contains a hawaiian island/word and establishment means is NA to 'endemic'. If a species contains any of these keywords in any of its names, it is most likely an endemic species. This should improve the accuracy of futher analyses, and any errors in this method would be far outweighed by its correct alterations.

In [2]:
#Ensure that no observations appear twice
data.drop_duplicates(subset='id')

#Proper capitalization of the preferred common name, and ensure taxon name follows binomial nomenclature capitalization format
data["preferred_common_name"]=data["preferred_common_name"].apply(lambda x: (' '.join(word.capitalize() for word in x.split(' '))) if isinstance(x, str) else x)
data["taxon_name"] = data["taxon_name"].apply(lambda x: x[0].upper()+x[1:])
data["conservation_status"] = data["conservation_status"].apply(lambda x: x if isinstance(x,str) else pd.NA)
data["observed_on"] = data["observed_on"].apply(lambda x: x if isinstance(x,str) else pd.NA)
data.replace("not specified",pd.NA, inplace = True)

#Change observations' establishment means that contain any of these keywords to "endemic"
keywords = ["Hawaii","Maui","Kauai","Hawaiʻi","Kauaʻi","Oʻahu","Oahu","Laysan", "Hawai'i", "Kaua'i", "O'ahu","Nahoa","Nihoa","Lanai","Lana'i","Lanaʻi","Niʻhoa","Ni'hoa","Molokai","Molokaʻi","Moloka'i"]
keyword = '|'.join(keywords)
data.loc[(
    (data["preferred_common_name"].str.contains(keyword, case = False)|data["species_guess"].str.contains(keyword,case = False)|data["taxon_name"].str.contains(keyword,case = False))
    & data["establishment_means"].isna())
    ,"establishment_means"] = "endemic"

<h3>Cross-referencing the GDIF dataset for Introduced taxa in Hawaii and  filling in all missing values for establishment_means</h3>
Citation: GBIF.org (19 February 2025) GBIF Occurrence Download https://doi.org/10.15468/dl.atsrcy

In [4]:
names = introduced_sp["scientificName"]
names = names.str.split(' ').str[0:2].str.join(' ')
data.loc[data["taxon_name"].isin(names)&data["establishment_means"].isna(),"establishment_means"] = "introduced"

Other Taxa that I manually found to be wrongly categorized as native but were present in the cross-referenced database: 

In [5]:
wrong_names = ["Lemna aequinoctialis","Tramea lacerata","Matricaria discoidea","Alternanthera sessilis"]
data.loc[data["taxon_name"].isin(wrong_names),"establishment_means"] = "introduced"
data["establishment_means"] = data["establishment_means"].apply(lambda x: x if isinstance(x,str) else pd.NA)

Altering some taxa I manually found as native, previously NA. Though most of the species marked as NA are probably native, I do not have the time to manually verify several thousand taxa. Therefore, altering the top 40 taxa with no establishment means, who comprised the majority of those observations, is good enough. <br><br> Upon further inspection, it would seem that the vast majority of these specimens are marine species. It would make sense that they are generally not officially recognized as native or endemic due to the fact that the oceans and marine ecosystems are far larger and less studied than terrestrial systems and especially isolated islands.


In [5]:
fish = (data.loc[data["taxon_rank"].str.contains("species")&data["establishment_means"].isna(),"taxon_name"].value_counts())[0:40].index
data.loc[data["taxon_name"].isin(fish),"establishment_means"]="native"



<h2>Save the cleaned Data</h2>

In [12]:
data.to_csv("CleanedData.csv", index = False)